In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

In [2]:
from cassandra.cluster import Cluster

cluster = Cluster(['cassandra'])  # Replace with your Cassandra host IP(s)
session = cluster.connect()

session.execute("""
    CREATE KEYSPACE IF NOT EXISTS my_keyspace
    WITH REPLICATION = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")
session.shutdown()
cluster.shutdown()

In [3]:
spark = SparkSession.builder \
    .appName("datamart") \
    .config("spark.sql.catalog.casscatalog", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.casscatalog.spark.cassandra.connection.host", "cassandra") \
    .getOrCreate()

26/04/02 16:59:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
def read_db(table_name):
    return spark.read.jdbc(url="jdbc:postgresql://postgres:5432/mydatabase",
                           table=table_name,
                           properties={
                               "user": "user",
                               "password": "password",
                               "driver": "org.postgresql.Driver"
                           })


def write_clickhouse(df, table_name):
    df.write \
        .format("clickhouse") \
        .option("host", "clickhouse") \
        .option("port", "8123") \
        .option("database", "default") \
        .option("user", "user") \
        .option("password", "password") \
        .option("table", table_name) \
        .option("order_by", "_id") \
        .mode("overwrite") \
        .save()


def write_cassandra(df, table_name):
    df.write \
        .format("org.apache.spark.sql.cassandra") \
        .option("spark.cassandra.connection.host", "cassandra") \
        .mode("append") \
        .partitionBy("_id") \
        .saveAsTable(f"casscatalog.my_keyspace.{table_name}")


def write_to_db(df, table_name, main_col="_id"):
    df2 = df.withColumn("_id", f.monotonically_increasing_id())
    # this is to properly order rows
    write_clickhouse(df2, table_name)
    write_cassandra(df2, table_name)

In [5]:
dim_products = read_db("dim_products")
dim_customers = read_db("dim_customers")
dim_sellers = read_db("dim_sellers")
dim_suppliers = read_db("dim_suppliers")
dim_stores = read_db("dim_stores")
fact_sales = read_db("fact_sales")

In [6]:
df_product_sales = fact_sales.join(dim_products,
                                   fact_sales.product_id == dim_products.id,
                                   "inner")

In [7]:
products_top_10 = df_product_sales.groupBy("name", "brand") \
    .agg(
        f.sum("sale_quantity").alias("total_sold"),
        f.sum("sale_total_price").alias("total_revenue")
    ) \
    .orderBy(f.col("total_sold").desc()) \
    .limit(10) \

write_to_db(products_top_10, "mart_products_top_10")
products_top_10.toPandas()

,name,brand,total_sold,total_revenue
0,Dog Food,Youspan,160,6161.230000000000000000
1,Bird Cage,Quinu,137,5294.800000000000000000
2,Bird Cage,Jayo,136,7755.600000000000000000
3,Cat Toy,Jayo,129,6417.820000000000000000
4,Bird Cage,Photobug,121,6561.390000000000000000
5,Bird Cage,Voomm,120,4728.120000000000000000
6,Bird Cage,Dynabox,120,6341.770000000000000000
7,Dog Food,Bluezoom,118,5935.430000000000000000
8,Cat Toy,Skimia,112,4796.380000000000000000
9,Cat Toy,Photobug,110,4782.210000000000000000


In [8]:
products_revenue_by_category = df_product_sales.groupBy("category") \
    .agg(f.sum("sale_total_price").alias("total_revenue")) \
    .orderBy(f.desc("total_revenue"))

write_to_db(products_revenue_by_category, "mart_products_revenue",
            "total_revenue")
products_revenue_by_category.toPandas()

,category,total_revenue
0,Toy,868101.630000000000000000
1,Cage,831117.940000000000000000
2,Food,830632.550000000000000000


In [9]:
products_rating = df_product_sales.groupBy("name", "brand") \
    .agg(
        f.avg("rating").alias("average_rating"),
        f.sum("reviews").alias("total_reviews")
    ) \
    .orderBy(f.col("average_rating").desc())
write_to_db(products_rating, "mart_products_rating", "average_rating")
products_rating.toPandas()

,name,brand,average_rating,total_reviews
0,Dog Food,Roombo,4.3800000000000000000000,2931
1,Bird Cage,Gevee,4.3666666666666666666667,2639
2,Bird Cage,Devbug,4.3500000000000000000000,1960
3,Cat Toy,Rhycero,4.3000000000000000000000,1610
4,Bird Cage,Flashset,4.2750000000000000000000,1530
...,...,...,...,...
1144,Bird Cage,Realpoint,1.7000000000000000000000,1866
1145,Bird Cage,Kwinu,1.7000000000000000000000,2323
1146,Dog Food,Topdrive,1.6444444444444444444444,3589
1147,Dog Food,Babbleset,1.6250000000000000000000,2307


In [10]:
df_customer_sales = fact_sales.join(dim_customers,
                                    fact_sales.customer_id == dim_customers.id,
                                    "inner")

In [11]:
customers_top_10 = df_customer_sales.groupBy("id","first_name", "last_name", "email") \
    .agg(f.sum("sale_total_price").alias("total_spent")) \
    .orderBy(f.desc("total_spent")) \
    .limit(10)
write_to_db(customers_top_10, "mart_customers_top_10", "total_spent")
customers_top_10.toPandas()

,id,first_name,last_name,email,total_spent
0,5157,Gus,Hartshorn,bfeasby57@youku.com,499.850000000000000000
1,2567,Hayes,McKain,sstappardbp@businessweek.com,499.800000000000000000
2,8667,Ava,Lomas,dsorea0@geocities.com,499.760000000000000000
3,8269,Dawna,Impey,rivattspm@un.org,499.760000000000000000
4,3402,Lavinia,Horsburgh,previllh3@tinyurl.com,499.730000000000000000
5,4977,Dame,Auchinleck,jthurnhamqe@sourceforge.net,499.710000000000000000
6,2389,Isahella,Colley,bselewayi0@chron.com,499.690000000000000000
7,9459,Sisely,Bonevant,wpulmano6@loc.gov,499.620000000000000000
8,2571,Nicky,Lattie,gcoupman2@bigcartel.com,499.620000000000000000
9,7347,Eran,Cotes,svispof9@t.co,499.590000000000000000


In [12]:
customers_by_country = dim_customers.groupBy("country") \
    .agg(f.count("id").alias("customer_count")) \
    .orderBy(f.desc("customer_count"))
write_to_db(customers_by_country, "mart_customers_by_country",
            "customer_count")
customers_by_country.toPandas()

,country,customer_count
0,China,1738
1,Indonesia,1174
2,Russia,628
3,Philippines,555
4,Brazil,385
...,...,...
199,Saint Martin,1
200,Seychelles,1
201,Belize,1
202,Guinea-Bissau,1


In [13]:
customers_average_check = df_customer_sales.groupBy("id","first_name", "last_name", "email") \
    .agg(f.avg("sale_total_price").alias("avg_check_amount"))
write_to_db(customers_average_check, "mart_customers_average_check", "id")
customers_average_check.toPandas()

,id,first_name,last_name,email,avg_check_amount
0,26,Carline,Pardoe,tbeasleyeh@so-net.ne.jp,362.8100000000000000000000
1,29,Cloris,Lisimore,kder9@latimes.com,69.5800000000000000000000
2,474,Lianna,Castilljo,bdurie3o@domainmarket.com,327.4500000000000000000000
3,964,Blinny,Saffell,nmoreinis8n@bing.com,209.3800000000000000000000
4,1677,Roseann,Papaminas,dlorenzot@a8.net,41.6400000000000000000000
...,...,...,...,...,...
9995,8826,Emmalyn,Sprague,sgaylor27@printfriendly.com,312.1700000000000000000000
9996,8954,Auria,Kilford,tnoon9k@jimdo.com,356.3700000000000000000000
9997,9663,Leland,Clampin,astidstonpf@harvard.edu,348.2800000000000000000000
9998,9772,Gonzales,Sedgmond,igammackl7@harvard.edu,145.3500000000000000000000


In [14]:
df_time_sales = fact_sales.withColumn("sale_year", f.year("sale_date")) \
                          .withColumn("sale_month", f.month("sale_date"))

In [15]:
time_trends = df_time_sales.groupBy("sale_year", "sale_month") \
    .agg(f.sum("sale_total_price").alias("total_revenue"), f.sum("sale_quantity").alias("total_items_sold")) \
    .orderBy("sale_year", "sale_month")
write_to_db(time_trends, "mart_time_monthly_yearly_trends", "sale_year")
time_trends.toPandas()

,sale_year,sale_month,total_revenue,total_items_sold
0,2021,1,224158.540000000000000000,4856
1,2021,2,192348.310000000000000000,4070
2,2021,3,207282.200000000000000000,4561
3,2021,4,206592.820000000000000000,4564
4,2021,5,211764.860000000000000000,4451
5,2021,6,215042.800000000000000000,4438
6,2021,7,220496.510000000000000000,4750
7,2021,8,221275.780000000000000000,4818
8,2021,9,210623.430000000000000000,4507
9,2021,10,228743.320000000000000000,4976


In [16]:
time_average_order_size = df_time_sales.groupBy("sale_year", "sale_month") \
    .agg(f.avg("sale_total_price").alias("avg_order_value")) \
    .orderBy("sale_year", "sale_month")
write_to_db(time_average_order_size, "mart_time_average_order_size",
            "sale_year")
time_average_order_size.toPandas()

,sale_year,sale_month,avg_order_value
0,2021,1,256.4743020594965675057208
1,2021,2,260.2818809201623815967524
2,2021,3,245.8863582443653618030842
3,2021,4,246.8253524492234169653524
4,2021,5,255.7546618357487922705314
5,2021,6,261.6092457420924574209246
6,2021,7,256.9889393939393939393939
7,2021,8,246.6842586399108138238573
8,2021,9,251.0410369487485101311085
9,2021,10,256.4386995515695067264574


In [17]:
df_store_sales = fact_sales.join(dim_stores,
                                 fact_sales.store_id == dim_stores.id, "left")

In [18]:
stores_top_5 = df_store_sales.groupBy("store_id", "name") \
    .agg(f.sum("sale_total_price").alias("total_revenue")) \
    .orderBy(f.desc("total_revenue")) \
    .limit(5)
write_to_db(stores_top_5, "mart_stores_top_5", "total_revenue")
stores_top_5.toPandas()

,store_id,name,total_revenue
0,4151,DabZ,499.850000000000000000
1,809,Thoughtblab,499.800000000000000000
2,3430,Camido,499.760000000000000000
3,3428,Edgeblab,499.760000000000000000
4,3249,Centizu,499.730000000000000000


In [19]:
stores_location = df_store_sales.groupBy("country", "city") \
    .agg(f.sum("sale_total_price").alias("total_revenue"), f.count("sale_id").alias("transactions_count")) \
    .orderBy(f.desc("transactions_count"))
write_to_db(stores_location, "mart_stores_location", "transactions_count")
stores_location.toPandas()

,country,city,total_revenue,transactions_count
0,China,Stockholm,2056.830000000000000000,8
1,China,Richmond,540.920000000000000000,3
2,China,Hats’avan,983.250000000000000000,3
3,China,Strasbourg,317.830000000000000000,3
4,China,Xinglong,1097.460000000000000000,3
...,...,...,...,...
9847,Peru,Emiliano Zapata,11.110000000000000000,1
9848,Ukraine,Long Beluah,465.300000000000000000,1
9849,Russia,Solosolo,180.350000000000000000,1
9850,China,Eci,194.890000000000000000,1


In [20]:
stores_average_check = df_store_sales.groupBy("store_id", "name") \
    .agg(f.avg("sale_total_price").alias("avg_check"))
write_to_db(stores_average_check, "mart_stores_average_check", "store_id")
stores_average_check.toPandas()

,store_id,name,avg_check
0,5409,Buzzdog,300.9000000000000000000000
1,4894,Thoughtbridge,456.9700000000000000000000
2,3764,Buzzster,15.1500000000000000000000
3,2040,Oyoba,147.2900000000000000000000
4,6721,Demimbu,324.3800000000000000000000
...,...,...,...
9995,6354,Wikizz,363.8100000000000000000000
9996,1326,Bluejam,12.5400000000000000000000
9997,7977,Browsecat,295.6100000000000000000000
9998,3045,Npath,433.5100000000000000000000


In [21]:
df_supplier_sales = fact_sales.join(dim_suppliers, fact_sales.supplier_id == dim_suppliers.id, "inner") \
                              .join(dim_products, fact_sales.product_id == dim_products.id, "inner")
df_supplier_sales.columns

['customer_id',
 'seller_id',
 'store_id',
 'supplier_id',
 'product_id',
 'pet_category',
 'sale_date',
 'sale_quantity',
 'sale_total_price',
 'sale_id',
 'name',
 'contact',
 'email',
 'phone',
 'address',
 'city',
 'country',
 'id',
 'brand',
 'category',
 'color',
 'description',
 'material',
 'name',
 'price',
 'quantity',
 'size',
 'weight',
 'release_data',
 'expiry_date',
 'reviews',
 'rating',
 'id']

In [22]:
suppliers_top_5 = df_supplier_sales.groupBy(fact_sales.supplier_id, dim_suppliers.name, dim_suppliers.email) \
    .agg(f.sum("sale_total_price").alias("total_revenue")) \
    .orderBy(f.desc("total_revenue")) \
    .limit(5)
write_to_db(suppliers_top_5, "mart_suppliers_top_5", "total_revenue")
suppliers_top_5.toPandas()

,supplier_id,name,email,total_revenue
0,2496,Brainverse,bfeasby57@ed.gov,499.850000000000000000
1,6262,Jamia,sstappardbp@webnode.com,499.800000000000000000
2,7292,Eabox,dsorea0@soup.io,499.760000000000000000
3,4225,Demimbu,rivattspm@nps.gov,499.760000000000000000
4,3923,Browsezoom,previllh3@pcworld.com,499.730000000000000000


In [23]:
suppliers_average_price = df_supplier_sales.groupBy(fact_sales.supplier_id, dim_suppliers.name, dim_suppliers.email) \
    .agg(f.avg("price").alias("avg_product_price"))
write_to_db(suppliers_average_price, "mart_suppliers_average_price",
            "supplier_id")
suppliers_average_price.toPandas()

,supplier_id,name,email,avg_product_price
0,26,Kwideo,hdenecamp9s@nyu.edu,52.0300000000000000000000
1,29,Einti,bmeadmorels@com.com,65.2300000000000000000000
2,474,Oyoyo,nclifftcc@marriott.com,91.6700000000000000000000
3,964,Oyoba,dbrierton8k@unesco.org,63.8000000000000000000000
4,1677,Realblab,dholworthmy@tuttocitta.it,52.9600000000000000000000
...,...,...,...,...
9995,8826,Omba,akyngeoa@google.com,19.5500000000000000000000
9996,8954,Skimia,hbrantii@archive.org,50.5700000000000000000000
9997,9663,Aivee,sstrickland7x@ft.com,53.4500000000000000000000
9998,9772,Buzzbean,tmccuthaisrg@smugmug.com,64.9400000000000000000000


In [24]:
suppliers_sale_by_country = df_supplier_sales.groupBy(dim_suppliers.country) \
    .agg(f.sum("sale_total_price").alias("total_revenue")) \
    .orderBy(f.desc("total_revenue"))
write_to_db(suppliers_sale_by_country, "mart_suppliers_sale_by_country",
            "total_revenue")
suppliers_sale_by_country.toPandas()

,country,total_revenue
0,China,492823.310000000000000000
1,Indonesia,265717.990000000000000000
2,Russia,149206.750000000000000000
3,Philippines,136135.100000000000000000
4,Brazil,97546.820000000000000000
...,...,...
198,Greenland,129.960000000000000000
199,Togo,103.910000000000000000
200,Aruba,90.490000000000000000
201,Grenada,60.540000000000000000


In [25]:
ratings_rating = products_rating.select("name", "brand", f.col("average_rating").alias("rating")) \
    .orderBy("average_rating")
least = ratings_rating.first()
most = ratings_rating.tail(1)[0]
ratings_rating_extremes = spark.createDataFrame([least, most])
write_to_db(ratings_rating_extremes, "mart_ratings_extremes", "rating")
ratings_rating_extremes.toPandas()

,name,brand,rating
0,Dog Food,Zoombox,1.200000000000000000
1,Dog Food,Roombo,4.380000000000000000


In [26]:
ratings_most_reviewed = products_rating.select(
    "brand", "name",
    f.col("total_reviews").alias("reviews")).orderBy(f.desc("reviews"))
write_to_db(ratings_most_reviewed, "mart_ratings_most_reviewed", "reviews")
ratings_most_reviewed.toPandas()

,brand,name,reviews
0,Youspan,Dog Food,13436
1,Quinu,Bird Cage,13061
2,Jayo,Cat Toy,12948
3,Voomm,Bird Cage,12050
4,Photobug,Cat Toy,11722
...,...,...,...
1144,Skyble,Cat Toy,606
1145,Layo,Cat Toy,483
1146,Quimba,Dog Food,460
1147,Eamia,Bird Cage,393


In [27]:
# Корреляция между рейтингом и объемом продаж
df_rating_sales = df_product_sales.groupBy("brand", "name") \
    .agg(
        f.avg("rating").alias("rating"),
        f.sum("sale_quantity").alias("sales")
    )
df_rating_sales.toPandas()

,brand,name,rating,sales
0,Devshare,Cat Toy,2.9125000000000000000000,39
1,Midel,Cat Toy,2.6750000000000000000000,59
2,Blogspan,Cat Toy,3.1400000000000000000000,28
3,Topicware,Dog Food,3.5500000000000000000000,33
4,Blogpad,Cat Toy,3.1777777777777777777778,57
...,...,...,...,...
1144,Youopia,Cat Toy,2.3200000000000000000000,30
1145,Pixope,Cat Toy,2.9538461538461538461538,73
1146,Yoveo,Dog Food,2.9285714285714285714286,40
1147,Mybuzz,Cat Toy,2.7000000000000000000000,29


In [28]:
ratings_correlation = df_rating_sales.select(
    f.corr("rating", "sales").alias("rating_sales_correlation"))
write_to_db(ratings_correlation, "mart_ratings_rating_sales_correlation",
            "rating_sales_correlation")
ratings_correlation.toPandas()

,rating_sales_correlation
0,-0.006764
